# 01 - Problem definition, Spark preprocessing and verification

In [1]:
from pathlib import Path
import sys, json, inspect
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "src/broadway").exists(), "Start in the project root or notebooks directory"
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown, Code
from broadway.config import (AUDIT, FEATURES, FIGURES, MODELS, TABLES, RAW,
                             PREDICTORS, NUMERIC, GRID, FOLDS)
from broadway.io import read_json, sha256
pd.set_option("display.max_columns", 12)
pd.set_option("display.precision", 4)


## Task 1 - Forecast contract and provenance

Predict next reported weekly gross, in nominal USD, for an observed continuing show
at a specific theatre. This is regression; four consecutive earlier reported weeks
are required. Prior-week reports are assumed available at prediction time; publication
timestamps and future schedules are absent. Revenue is not profit or an intervention outcome.

Source: [TidyTuesday, 28 April 2020](https://github.com/rfordatascience/tidytuesday/tree/main/data/2020/2020-04-28),



In [2]:
REBUILD_PREPROCESSING = False
if REBUILD_PREPROCESSING:
    if (TABLES / "final_evaluation_receipt.json").exists():
        raise RuntimeError("Final evaluation exists; preserve the evaluated inputs.")
    from broadway.spark_pipeline import acquire, start_spark, smoke, preprocess
    from broadway.io import write_json
    spark = start_spark()
    try:
        write_json(AUDIT / "spark_smoke.json", smoke(spark, AUDIT / "smoke.parquet"))
        preprocess(spark, acquire())
    finally:
        spark.stop()
    print("Rebuilt with Spark. Rerun development validation to freeze the new data fingerprint.")
else:
    assert FEATURES.exists(), "No processed data: enable rebuild after environment setup."
    print("Review mode: preserved Spark outputs; no ingestion or feature rebuild in this execution.")
manifest = read_json(AUDIT / "input_manifest.json")
assert sha256(RAW) == manifest["sha256"], "Raw bytes differ from the recorded manifest"
display(pd.Series({k: manifest[k] for k in ["raw_rows", "raw_columns", "first_week",
    "last_week", "bytes", "eligible_rows", "sha256"]}, name="Recorded source inventory"))
display(read_json(RAW.with_suffix(".source.json")))

Review mode: preserved Spark outputs; no ingestion or feature rebuild in this execution.


raw_rows                                                     47524
raw_columns                                                     14
first_week                                              1985-06-09
last_week                                               2020-03-01
bytes                                                      4904731
eligible_rows                                                42039
sha256           1e383807947d11121737abe21c3b134011b878c5bd8a8c...
Name: Recorded source inventory, dtype: object

{'source_url': 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2020/2020-04-28/grosses.csv',
 'method': 'downloaded from source URL',
 'acquired_at_utc': '2026-11-16T15:20:34.505061+00:00',
 'sha256': '1e383807947d11121737abe21c3b134011b878c5bd8a8cc4c6bc4e7cc7b04e4d'}

## Task 2.1 - Tools, schema and scale

Spark performs actual ingestion and engineering. pandas here displays saved audits
and reads the engineered handoff; it does not implement another raw-data pipeline.
This roughly 4.9 MB CSV fits locally. 

In [3]:
display(pd.DataFrame(manifest["raw_schema"]["fields"])[["name", "type", "nullable"]])
display(pd.Series(read_json(AUDIT / "spark_smoke.json"), name="Recorded runtime / filesystem check"))

,name,type,nullable
0,week_ending,date,True
1,week_number,integer,True
2,weekly_gross_overall,double,True
3,show,string,True
4,theatre,string,True
5,weekly_gross,double,True
6,potential_gross,double,True
7,avg_ticket_price,double,True
8,top_ticket_price,double,True
9,seats_sold,integer,True


spark                                         4.0.3
java                                        17.0.12
hadoop                                        3.4.1
hadoop_home            D:\3rd Year\DS\.tools\hadoop
parquet_rows                                      5
value_sum                                        20
verified_at_utc    2026-11-16T15:32:17.007564+00:00
python                                       3.13.5
platform                  Windows-11-10.0.26300-SP0
pyspark                                       4.0.3
pandas                                        2.3.3
pyarrow                                      22.0.0
numpy                                         2.2.6
scipy                                        1.16.3
scikit-learn                                  1.7.2
matplotlib                                   3.10.7
pytest                                        8.4.2
nbformat                                     5.10.4
nbclient                                     0.10.2
ipykernel   

## Task 2.2 - Cleaning and missing values

Missing keys/targets, negative/nonfinite gross and non-Sunday dates are excluded with
reasons; targets are never imputed. 


In [4]:
for filename in ["row_counts.csv", "missingness.csv", "excluded_rows.csv", "malformed_csv_structure.csv"]:
    display(Markdown(f"**{filename}**"))
    display(pd.read_csv(AUDIT / filename))
display(read_json(AUDIT / "duplicates.json"))
display(read_json(AUDIT / "parse_audit.json"))

**row_counts.csv**

,stage,rows
0,raw,47524
1,exact_unique,47524
2,excluded_invalid_key_date_or_target,0
3,cleaned,47524
4,engineered_all,47524
5,eligible,42039
6,ineligible_retained,5485


**missingness.csv**

,field,null_count,nan_count,infinite_count,missing_count,missing_rate
0,week_ending,0,0,0,0,0.0000
1,week_number,0,0,0,0,0.0000
2,weekly_gross_overall,0,0,0,0,0.0000
3,show,0,0,0,0,0.0000
4,theatre,0,0,0,0,0.0000
5,weekly_gross,0,0,0,0,0.0000
6,potential_gross,12613,0,0,12613,0.2654
7,avg_ticket_price,0,0,0,0,0.0000
8,top_ticket_price,11357,0,0,11357,0.2390
9,seats_sold,0,0,0,0,0.0000


**excluded_rows.csv**

,week_ending,week_number,weekly_gross_overall,show,theatre,weekly_gross,...,seats_sold,seats_in_theatre,pct_capacity,performances,previews,exclusion_reason


**malformed_csv_structure.csv**

,record,field_count,date_token


{'raw_rows': 47524,
 'exact_duplicates_removed': 0,
 'conflicting_keys': 0,
 'conflicting_key_examples': []}

{'csv_records': 47524,
 'spark_rows': 47524,
 'malformed_rows': 0,
 'malformed_examples': []}

## Task 2.3 - Outliers: flag, investigate, retain

Zero gross, zero seating capacity, occupancy above 100% and large gross are retained
review flags. The high-gross threshold is Q3 + 3 IQR estimated through 2014, not an
automatic exclusion.


In [5]:
display(read_json(AUDIT / "anomaly_policy.json"))
display(pd.read_csv(AUDIT / "anomaly_counts.csv"))
examples = pd.read_csv(AUDIT / "anomaly_examples_through_2014.csv")
display(examples.groupby("flag", sort=False).head(2))

{'gross_review_threshold_nominal_usd': 1906208.0,
 'threshold_source': 'Q3 + 3 IQR on cleaned target weeks through 2014; approximate quantiles error=0.001',
 'zero_and_overcapacity_policy': 'Retain; >1 occupancy can represent standing-room sales.',
 'optional_nonfinite_policy': 'NaN/infinity converted to null after raw missingness audit.',
 'examples_scope': 'through 2014 only; aggregate quality counts include all dates'}

,flag,count,action
0,zero_reported_gross,125,retained for inspection
1,zero_theatre_capacity,86,retained for inspection
2,occupancy_above_100_pct,4572,retained for inspection
3,high_gross_development_3iqr,827,retained for inspection
4,negative_reported_non_target,0,retained for inspection


,flag,week_ending,week_number,weekly_gross_overall,show,theatre,...,top_ticket_price,seats_sold,seats_in_theatre,pct_capacity,performances,previews
0,zero_reported_gross,1986-03-30,43,3.9917e+06,"""The Flying Karamazov Brothers """"Juggling and ...",Vivian Beaumont Theater,...,NaN,0,1105,0.0000,0,4
1,zero_reported_gross,1986-04-06,44,3.6327e+06,"""The Flying Karamazov Brothers """"Juggling and ...",Vivian Beaumont Theater,...,NaN,0,1105,0.0000,8,0
12,zero_theatre_capacity,2007-11-18,25,2.9338e+06,A Bronx Tale,Walter Kerr Theatre,...,251.50,0,0,0.0000,0,0
13,zero_theatre_capacity,2007-11-25,26,4.2980e+06,A Bronx Tale,Walter Kerr Theatre,...,251.50,0,0,0.0000,0,0
24,occupancy_above_100_pct,1989-08-06,10,4.3965e+06,"""Mandy Patinkin in Concert: """"Dress Casual""""""",Helen Hayes Theatre,...,NaN,3494,499,1.0003,7,0
25,occupancy_above_100_pct,2004-12-19,29,1.5944e+07,700 Sundays,Broadhurst Theatre,...,101.25,8386,1186,1.0101,7,0
36,high_gross_development_3iqr,2014-12-28,31,4.0994e+07,Aladdin,New Amsterdam Theatre,...,252.50,15508,1723,1.0001,9,0
37,high_gross_development_3iqr,2012-12-30,31,3.7441e+07,Annie,Palace Theatre,...,250.00,14862,1708,0.9668,9,0


## Task 2.4 - Historical features and leakage prevention

Runs reset after gaps other than seven days or theatre changes. Lags stay inside
show–theatre–run groups. `rowsBetween(-4, -1)` excludes the target and uses four earlier
rows. Known run age is observed history, not official production age. 


In [6]:
from broadway.spark_pipeline import engineer_features
display(pd.DataFrame({"predictor": PREDICTORS,
    "type": ["numeric" if p in NUMERIC else "categorical" for p in PREDICTORS]}))
display(Code(inspect.getsource(engineer_features), language="python"))

,predictor,type
0,weekly_gross_lag1,numeric
1,seats_sold_lag1,numeric
2,pct_capacity_lag1,numeric
3,performances_lag1,numeric
4,previews_lag1,numeric
5,gross_prior_4_mean,numeric
6,run_weeks_known,numeric
7,theatre,categorical
8,month,categorical


def engineer_features(clean):
    pair = Window.partitionBy("show", "theatre").orderBy("week_ending")
    previous = F.lag("week_ending").over(pair)
    frame = clean.withColumn("_run_start", F.when(previous.isNull() | (F.datediff("week_ending", previous) != 7), 1).otherwise(0))
    frame = frame.withColumn("inferred_run", F.sum("_run_start").over(pair.rowsBetween(Window.unboundedPreceding, 0)))
    run = Window.partitionBy("show", "theatre", "inferred_run").orderBy("week_ending")
    for col in ["weekly_gross", "seats_sold", "pct_capacity", "performances", "previews"]:
        frame = frame.withColumn(col + "_lag1", F.lag(col).over(run))
    history = run.rowsBetween(-4, -1)
    frame = (frame.withColumn("gross_prior_4_mean", F.avg("weekly_gross").over(history))
             .withColumn("prior_gross_count", F.count("weekly_gross").over(history))
             .withColumn("run_weeks_known", F.row_number().over(run) - 1)
             .withColumn("month", F.month("week_ending").cast("string"))
             .withColumn("year", F.year("week_ending")))
    return frame.withColumn("eligible", (F.col("run_weeks_known") >= 4) & (F.col("prior_gross_count") == 4)).drop("_run_start")

## Task 2.5 — Worked example and export verification

Recalculate the recorded four-week mean, then compare it to the exported row.



In [7]:
import pyarrow.parquet as pq
example = read_json(AUDIT / "leakage_example.json")
history = pd.DataFrame(example["prior_dates_and_grosses"])
display(history)
calculated = history.weekly_gross.mean()
assert np.isclose(calculated, example["model_feature_gross_prior_4_mean"])
early = pd.read_parquet(FEATURES, engine="pyarrow", filters=[("year", "<=", 2014)])
match = early.loc[(early.show == example["show"]) & (early.theatre == example["theatre"])
                  & (pd.to_datetime(early.week_ending) == pd.Timestamp(example["target_date"]))]
assert len(match) == 1 and np.isclose(match.gross_prior_4_mean.iloc[0], calculated)
print(f"Verified prior-four-week mean: ${calculated:,.2f}; target: {example['target_date']}")
display(early.sort_values(["week_ending", "show", "theatre"])[
    ["show", "week_ending", "weekly_gross", "weekly_gross_lag1",
     "gross_prior_4_mean", "run_weeks_known", "eligible"]].head(12))
print(pq.ParquetDataset(FEATURES).schema)
display(read_json(AUDIT / "parquet_roundtrip.json"))

,week_ending,weekly_gross
0,1985-06-09,282368.0
1,1985-06-16,301417.0
2,1985-06-23,296438.0
3,1985-06-30,292841.0


Verified prior-four-week mean: $293,266.00; target: 1985-07-07


,show,week_ending,weekly_gross,weekly_gross_lag1,gross_prior_4_mean,run_weeks_known,eligible
370,42nd Street,1985-06-09,282368.0,NaN,NaN,0,False
791,A Chorus Line,1985-06-09,222584.0,NaN,NaN,0,False
3136,Aren't We All?,1985-06-09,249272.0,NaN,NaN,0,False
3143,Arms and the Man,1985-06-09,95688.0,NaN,NaN,0,False
3269,As Is,1985-06-09,61059.0,NaN,NaN,0,False
4816,Big River,1985-06-09,255386.0,NaN,NaN,0,False
5110,Biloxi Blues,1985-06-09,306839.0,NaN,NaN,0,False
5671,Brighton Beach Memoirs,1985-06-09,107392.0,NaN,NaN,0,False
6874,Cats,1985-06-09,461880.0,NaN,NaN,0,False
10474,Doubles,1985-06-09,47452.0,NaN,NaN,0,False


week_ending: date32[day]
week_number: int32
weekly_gross_overall: double
show: string
theatre: string
weekly_gross: double
potential_gross: double
avg_ticket_price: double
top_ticket_price: double
seats_sold: int32
seats_in_theatre: int32
pct_capacity: double
performances: int32
previews: int32
inferred_run: int64
weekly_gross_lag1: double
seats_sold_lag1: int32
pct_capacity_lag1: double
performances_lag1: int32
previews_lag1: int32
gross_prior_4_mean: double
prior_gross_count: int64 not null
run_weeks_known: int32 not null
month: string
year: int32
eligible: bool not null
-- schema metadata --
org.apache.spark.version: '4.0.3'
org.apache.spark.sql.parquet.row.metadata: '{"type":"struct","fields":[{"' + 1843


{'rows': 47524,
 'schema': [['week_ending', 'date'],
  ['week_number', 'int'],
  ['weekly_gross_overall', 'double'],
  ['show', 'string'],
  ['theatre', 'string'],
  ['weekly_gross', 'double'],
  ['potential_gross', 'double'],
  ['avg_ticket_price', 'double'],
  ['top_ticket_price', 'double'],
  ['seats_sold', 'int'],
  ['seats_in_theatre', 'int'],
  ['pct_capacity', 'double'],
  ['performances', 'int'],
  ['previews', 'int'],
  ['inferred_run', 'bigint'],
  ['weekly_gross_lag1', 'double'],
  ['seats_sold_lag1', 'int'],
  ['pct_capacity_lag1', 'double'],
  ['performances_lag1', 'int'],
  ['previews_lag1', 'int'],
  ['gross_prior_4_mean', 'double'],
  ['prior_gross_count', 'bigint'],
  ['run_weeks_known', 'int'],
  ['month', 'string'],
  ['year', 'int'],
  ['eligible', 'boolean']],
 'count_matches': True,
 'types_match': True,
 'row_multisets_match': True,
 'nullable_note': 'Parquet may mark non-null computed fields nullable on read.'}